# LeetCode 1358 — Number of Substrings Containing All Three Characters bằng Python

Link đề: https://leetcode.com/problems/number-of-substrings-containing-all-three-characters/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def numberOfSubstrings(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu cách **đếm chuỗi con theo vị trí** (mỗi cặp `(i, j)` là một chuỗi con, kể cả khi nội dung trùng nhau).
2. Nhận ra **tính đơn điệu**: nếu `s[i..j]` chứa đủ `a, b, c` thì mọi `s[i..j']` với `j' ≥ j` cũng chứa đủ ⇒ đếm theo khối `n − j`.
3. Đi từ brute force `O(n²)` → **cửa sổ trượt** `O(n)` → **vị trí xuất hiện cuối** của `a/b/c` (cộng `min(last) + 1`) → **bù trừ (inclusion–exclusion)**.
4. Biết hai kiểu đếm của cửa sổ trượt: "cố định trái, đếm phải" và "cố định phải, đếm trái".
5. Tránh lỗi: khởi tạo `last` bằng 0, quên `+1`, đếm `right − left + 1`, đếm chuỗi con phân biệt.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm các ký tự `a`, `b`, `c`. Trả về **số chuỗi con** (đoạn liên tiếp) chứa **ít nhất một** lần xuất hiện của **cả ba** ký tự `a`, `b`, `c`.

**Ví dụ 1:** `s = "abcabc"` → `10` (các chuỗi con: `"abc"`, `"abca"`, `"abcab"`, `"abcabc"`, `"bca"`, `"bcab"`, `"bcabc"`, `"cab"`, `"cabc"` và `"abc"` (lần nữa, ở vị trí khác)).

**Ví dụ 2:** `s = "aaacb"` → `3` (`"aaacb"`, `"aacb"`, `"acb"`).

**Ví dụ 3:** `s = "abc"` → `1`.

**Ràng buộc:** `3 <= s.length <= 5·10^4`; `s` chỉ gồm `'a'`, `'b'`, `'c'`.

In [1]:
import random
import time

examples = [
    ("abcabc", 10),
    ("aaacb", 3),
    ("abc", 1),
]
for s, want in examples:
    print(f"s={s!r:10} -> mong đợi {want} | tổng số chuỗi con = n(n+1)/2 = {len(s) * (len(s) + 1) // 2}")

s='abcabc'   -> mong đợi 10 | tổng số chuỗi con = n(n+1)/2 = 21
s='aaacb'    -> mong đợi 3 | tổng số chuỗi con = n(n+1)/2 = 15
s='abc'      -> mong đợi 1 | tổng số chuỗi con = n(n+1)/2 = 6


## 3. Kiến thức nền

### 3.1 Chuỗi con và số lượng chuỗi con

Chuỗi con `s[i:j+1]` xác định bởi cặp `0 ≤ i ≤ j < n`. Có `n(n+1)/2` chuỗi con. Hai chuỗi con ở **vị trí khác nhau** được đếm riêng dù nội dung giống nhau.

In [2]:
s = "abca"
subs = [s[i:j + 1] for i in range(len(s)) for j in range(i, len(s))]
print(len(subs), subs)

10 ['a', 'ab', 'abc', 'abca', 'b', 'bc', 'bca', 'c', 'ca', 'a']


### 3.2 `set` và kiểm tra "đủ ba ký tự"

`set(t) == {'a','b','c'}` hoặc `len(set(t)) == 3` (vì chỉ có 3 loại ký tự).

In [3]:
for t in ["abc", "aab", "cabbb"]:
    print(t, set(t), len(set(t)) == 3)

abc {'a', 'c', 'b'} True
aab {'a', 'b'} False
cabbb {'a', 'c', 'b'} True


### 3.3 Mảng đếm 3 phần tử

Ánh xạ `'a'→0, 'b'→1, 'c'→2` bằng `ord(ch) - ord('a')`. Cửa sổ "đủ" ⇔ cả ba ô đều `> 0`.

In [4]:
cnt = [0, 0, 0]
for ch in "aacb":
    cnt[ord(ch) - 97] += 1
print(cnt, "đủ?", all(cnt))

[2, 1, 1] đủ? True


### 3.4 Tính đơn điệu khi mở rộng chuỗi con

Nếu `s[i..j]` đã chứa đủ `a, b, c` thì `s[i..j+1]`, `s[i..j+2]`, … cũng chứa đủ (thêm ký tự không làm mất ký tự nào). Tương tự với việc kéo dài sang trái. Đây là nền tảng của mọi lời giải nhanh.

In [5]:
s = "aaacb"
i = 1
for j in range(i, len(s)):
    print(f"s[{i}..{j}] = {s[i:j + 1]!r:7} đủ? {len(set(s[i:j + 1])) == 3}")

s[1..1] = 'a'     đủ? False
s[1..2] = 'aa'    đủ? False
s[1..3] = 'aac'   đủ? False
s[1..4] = 'aacb'  đủ? True


### 3.5 Công thức `L(L+1)/2` cho một đoạn liên tiếp

Một đoạn dài `L` có `L(L+1)/2` chuỗi con. Dùng cho lời giải bù trừ: đếm chuỗi con **không chứa** một tập ký tự bằng cách chia `s` thành các đoạn tối đa không chứa chúng.

In [6]:
import re
s = "abcaab"
runs = [len(m) for m in re.split("c", s)]
print("các đoạn không chứa 'c':", re.split("c", s), "-> số chuỗi con không chứa 'c' =", sum(L * (L + 1) // 2 for L in runs))

các đoạn không chứa 'c': ['ab', 'aab'] -> số chuỗi con không chứa 'c' = 9


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `s = "abcabc"` (n = 6) theo kiểu "cố định đầu trái `i`, tìm đầu phải nhỏ nhất `j` sao cho `s[i..j]` đủ ba ký tự":

| i | j nhỏ nhất | chuỗi con nhỏ nhất | số chuỗi con hợp lệ bắt đầu ở i = n − j |
|---|---|---|---|
| 0 | 2 | abc | 4 |
| 1 | 3 | bca | 3 |
| 2 | 4 | cab | 2 |
| 3 | 5 | abc | 1 |
| 4 | – | (không có) | 0 |
| 5 | – | (không có) | 0 |

Tổng `4 + 3 + 2 + 1 = 10` ✔.

**Cách nhìn ngược lại** (cố định đầu phải `j`): số đầu trái hợp lệ = `min(vị trí cuối của a, b, c) + 1`.

In [7]:
s = "abcabc"
n = len(s)
total = 0
for i in range(n):
    for j in range(i, n):
        if len(set(s[i:j + 1])) == 3:
            print(f"i={i}: j nhỏ nhất = {j} ({s[i:j + 1]!r}) -> cộng {n - j}")
            total += n - j
            break
print("tổng =", total)

i=0: j nhỏ nhất = 2 ('abc') -> cộng 4
i=1: j nhỏ nhất = 3 ('bca') -> cộng 3
i=2: j nhỏ nhất = 4 ('cab') -> cộng 2
i=3: j nhỏ nhất = 5 ('abc') -> cộng 1
tổng = 10


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi đầu trái `i`, mở rộng `j` và cập nhật mảng đếm. Khi lần đầu đủ ba ký tự, nhờ đơn điệu, cộng ngay `n − j` rồi `break`.

(Phiên bản "thô" hơn nữa là duyệt mọi `(i, j)` và gọi `set` — `O(n³)` — dùng làm trọng tài trong bộ test.)

In [8]:
def count_brute(s):
    n = len(s)
    ans = 0
    for i in range(n):
        cnt = [0, 0, 0]
        for j in range(i, n):
            cnt[ord(s[j]) - 97] += 1
            if cnt[0] and cnt[1] and cnt[2]:
                ans += n - j          # s[i..j], s[i..j+1], ..., s[i..n-1]
                break
    return ans

def count_naive(s):
    """O(n^3): kiểm tra từng chuỗi con bằng set."""
    n = len(s)
    return sum(len(set(s[i:j + 1])) == 3 for i in range(n) for j in range(i, n))

for s, want in examples:
    print(f"{s!r:10} -> {count_brute(s)}, {count_naive(s)} (mong đợi {want})")

'abcabc'   -> 10, 10 (mong đợi 10)
'aaacb'    -> 3, 3 (mong đợi 3)
'abc'      -> 1, 1 (mong đợi 1)


**Độ phức tạp:** `O(n²)` worst (ví dụ `"aaaa…abc"`: mỗi `i` phải quét tới cuối). Với `n = 5·10^4` là ~`1,25·10^9` bước — quá chậm.

**Nút thắt:** khi tăng `i` lên `i+1`, ta **quét lại** từ `j = i+1`, trong khi `j` nhỏ nhất của `i+1` không bao giờ nhỏ hơn `j` của `i` (đơn điệu!). ⇒ Đầu phải chỉ cần đi tiến ⇒ **hai con trỏ**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (cửa sổ trượt):** gọi `j(i)` là đầu phải nhỏ nhất làm `s[i..j]` đủ. Khi `i` tăng thì `j(i)` **không giảm**. ⇒ Hai con trỏ cùng tiến, mỗi con trỏ đi tối đa `n` bước.

**Insight 2 (vị trí cuối):** cố định đầu phải `j`. Gọi `last[x]` là vị trí xuất hiện **gần nhất (≤ j)** của ký tự `x`. `s[i..j]` đủ ba ký tự ⇔ `i ≤ last[a]`, `i ≤ last[b]`, `i ≤ last[c]` ⇔ `i ≤ min(last)`. Số lựa chọn `i ∈ [0, min(last)]` là `min(last) + 1` (nếu chưa thấy đủ, `min(last) = −1` ⇒ cộng 0).

**Quy tắc nếu… thì…:**
- Nếu cửa sổ `[left, right]` đã đủ thì mọi chuỗi con bắt đầu tại `left` và kết thúc từ `right` trở đi đều hợp lệ ⇒ cộng `n − right`, rồi thu hẹp `left`.
- Với mỗi `right`: cập nhật `last[s[right]] = right`, cộng `min(last) + 1`.

**Insight 3 (bù trừ):** `#đủ = tổng − #thiếu a − #thiếu b − #thiếu c + #thiếu {a,b} + #thiếu {a,c} + #thiếu {b,c} − #thiếu {a,b,c}`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cửa sổ trượt (cố định trái, đếm về phía phải)

**Giải thích từng đoạn:**
1. `right` chạy từ trái sang phải, thêm `s[right]` vào mảng đếm.
2. Trong khi cửa sổ `[left, right]` đủ ba ký tự: mọi chuỗi con `s[left..right']` với `right' ≥ right` hợp lệ ⇒ `ans += n − right`; bỏ `s[left]`, `left += 1`.
3. Mỗi `left` được "chốt" đúng một lần với `right` nhỏ nhất của nó.

In [9]:
def count_window(s):
    n = len(s)
    cnt = [0, 0, 0]
    ans = left = 0
    for right in range(n):
        cnt[ord(s[right]) - 97] += 1
        while cnt[0] and cnt[1] and cnt[2]:
            ans += n - right
            cnt[ord(s[left]) - 97] -= 1
            left += 1
    return ans

def count_window_left(s):
    """Biến thể: cố định phải, sau khi thu hẹp thì mọi đầu trái < left đều hợp lệ."""
    cnt = [0, 0, 0]
    ans = left = 0
    for ch in s:
        cnt[ord(ch) - 97] += 1
        while cnt[0] and cnt[1] and cnt[2]:
            cnt[ord(s[left]) - 97] -= 1
            left += 1
        ans += left
    return ans

for s, want in examples:
    print(f"{s!r:10} -> {count_window(s)}, {count_window_left(s)} (mong đợi {want})")

'abcabc'   -> 10, 10 (mong đợi 10)
'aaacb'    -> 3, 3 (mong đợi 3)
'abc'      -> 1, 1 (mong đợi 1)


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def numberOfSubstrings(self, s: str) -> int:
        n = len(s)
        cnt = [0, 0, 0]
        ans = left = 0
        for right in range(n):
            cnt[ord(s[right]) - 97] += 1
            while cnt[0] and cnt[1] and cnt[2]:
                ans += n - right
                cnt[ord(s[left]) - 97] -= 1
                left += 1
        return ans

print(Solution().numberOfSubstrings("abcabc"))

10


**Độ phức tạp:** `O(n)` (mỗi con trỏ đi tối đa `n` bước), bộ nhớ `O(1)`.

### 7.2 Cách 3 — Vị trí xuất hiện cuối của `a/b/c`

**Giải thích:** `last = [-1, -1, -1]`. Với mỗi `j`: `last[s[j]] = j`, cộng `min(last) + 1`. Không cần vòng `while` — mỗi bước `O(1)`. Đây là lời giải ngắn và đẹp nhất.

In [11]:
def count_last(s):
    last = [-1, -1, -1]
    ans = 0
    for j, ch in enumerate(s):
        last[ord(ch) - 97] = j
        ans += min(last) + 1
    return ans

for s, want in examples:
    print(f"{s!r:10} -> {count_last(s)} (mong đợi {want})")

'abcabc'   -> 10 (mong đợi 10)
'aaacb'    -> 3 (mong đợi 3)
'abc'      -> 1 (mong đợi 1)


Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    def numberOfSubstrings(self, s: str) -> int:
        last = [-1, -1, -1]
        ans = 0
        for j, ch in enumerate(s):
            last[ord(ch) - 97] = j
            ans += min(last) + 1
        return ans

print(Solution().numberOfSubstrings("aaacb"))

3


**Độ phức tạp:** `O(n)`, bộ nhớ `O(1)`.

### 7.3 Cách 4 — Vị trí xuất hiện **kế tiếp** (theo gợi ý của đề)

**Giải thích:** duyệt từ phải sang trái, giữ `nxt[x]` = vị trí đầu tiên `≥ i` của ký tự `x` (mặc định `n` nếu không có). Với đầu trái `i`, đầu phải nhỏ nhất là `max(nxt)`; số chuỗi con hợp lệ bắt đầu tại `i` là `n − max(nxt)` (bằng 0 khi `max(nxt) = n`). Đây là "gương" của Cách 3.

In [13]:
def count_next(s):
    n = len(s)
    nxt = [n, n, n]
    ans = 0
    for i in range(n - 1, -1, -1):
        nxt[ord(s[i]) - 97] = i
        ans += n - max(nxt)
    return ans

for s, want in examples:
    print(f"{s!r:10} -> {count_next(s)} (mong đợi {want})")

'abcabc'   -> 10 (mong đợi 10)
'aaacb'    -> 3 (mong đợi 3)
'abc'      -> 1 (mong đợi 1)


### 7.4 Cách 5 — Bù trừ (inclusion–exclusion)

**Giải thích:** đặt `N(S)` = số chuỗi con **không chứa ký tự nào trong tập `S`**. Chia `s` thành các đoạn tối đa không chứa ký tự của `S`, mỗi đoạn dài `L` góp `L(L+1)/2`. Khi đó

`đáp án = N(∅) − N({a}) − N({b}) − N({c}) + N({a,b}) + N({a,c}) + N({b,c}) − N({a,b,c})`,

với `N(∅) = n(n+1)/2`, `N({a,b,c}) = 0`. Cách này không cần cửa sổ, và tổng quát hóa được cho tập ký tự bất kỳ.

In [14]:
from itertools import combinations

def avoid_count(s, banned):
    total = run = 0
    for ch in s:
        if ch in banned:
            total += run * (run + 1) // 2
            run = 0
        else:
            run += 1
    return total + run * (run + 1) // 2

def count_incl_excl(s):
    letters = "abc"
    ans = 0
    for k in range(len(letters) + 1):
        for S in combinations(letters, k):
            ans += (-1) ** k * avoid_count(s, set(S))
    return ans

for s, want in examples:
    print(f"{s!r:10} -> {count_incl_excl(s)} (mong đợi {want})")

'abcabc'   -> 10 (mong đợi 10)
'aaacb'    -> 3 (mong đợi 3)
'abc'      -> 1 (mong đợi 1)


**Độ phức tạp:** `2^3 = 8` lần quét ⇒ `O(8n) = O(n)`, bộ nhớ `O(1)`.

## 8. TRACE từng bước

### 8.1 Trace Cách 3 trên `s = "abcabc"`

| j | ký tự | last (a, b, c) | min(last) + 1 | ans |
|---|---|---|---|---|
| 0 | a | (0, −1, −1) | 0 | 0 |
| 1 | b | (0, 1, −1) | 0 | 0 |
| 2 | c | (0, 1, 2) | 1 | 1 |
| 3 | a | (3, 1, 2) | 2 | 3 |
| 4 | b | (3, 4, 2) | 3 | 6 |
| 5 | c | (3, 4, 5) | 4 | 10 |

In [15]:
def trace_last(s):
    last = [-1, -1, -1]
    ans = 0
    print(" j | ch | last(a,b,c)  | +min+1 | ans")
    for j, ch in enumerate(s):
        last[ord(ch) - 97] = j
        add = min(last) + 1
        ans += add
        print(f"{j:2d} |  {ch} | {str(tuple(last)):12} | {add:6d} | {ans}")
    return ans

print("Kết quả:", trace_last("abcabc"))
print()
print("Kết quả:", trace_last("aaacb"))

 j | ch | last(a,b,c)  | +min+1 | ans
 0 |  a | (0, -1, -1)  |      0 | 0
 1 |  b | (0, 1, -1)   |      0 | 0
 2 |  c | (0, 1, 2)    |      1 | 1
 3 |  a | (3, 1, 2)    |      2 | 3
 4 |  b | (3, 4, 2)    |      3 | 6
 5 |  c | (3, 4, 5)    |      4 | 10
Kết quả: 10

 j | ch | last(a,b,c)  | +min+1 | ans
 0 |  a | (0, -1, -1)  |      0 | 0
 1 |  a | (1, -1, -1)  |      0 | 0
 2 |  a | (2, -1, -1)  |      0 | 0
 3 |  c | (2, -1, 3)   |      0 | 0
 4 |  b | (2, 4, 3)    |      3 | 3
Kết quả: 3


### 8.2 Trace Cách 2 (cửa sổ trượt) trên `s = "aaacb"`

In [16]:
def trace_window(s):
    n = len(s)
    cnt = [0, 0, 0]
    ans = left = 0
    for right in range(n):
        cnt[ord(s[right]) - 97] += 1
        print(f"right={right} thêm {s[right]!r}: cửa sổ={s[left:right + 1]!r}, cnt={cnt}")
        while all(cnt):
            ans += n - right
            print(f"    đủ! cửa sổ [{left},{right}]={s[left:right + 1]!r} -> cộng n-right={n - right}, ans={ans}; bỏ {s[left]!r}")
            cnt[ord(s[left]) - 97] -= 1
            left += 1
    return ans

print("Kết quả:", trace_window("aaacb"))

right=0 thêm 'a': cửa sổ='a', cnt=[1, 0, 0]
right=1 thêm 'a': cửa sổ='aa', cnt=[2, 0, 0]
right=2 thêm 'a': cửa sổ='aaa', cnt=[3, 0, 0]
right=3 thêm 'c': cửa sổ='aaac', cnt=[3, 0, 1]
right=4 thêm 'b': cửa sổ='aaacb', cnt=[3, 1, 1]
    đủ! cửa sổ [0,4]='aaacb' -> cộng n-right=1, ans=1; bỏ 'a'
    đủ! cửa sổ [1,4]='aacb' -> cộng n-right=1, ans=2; bỏ 'a'
    đủ! cửa sổ [2,4]='acb' -> cộng n-right=1, ans=3; bỏ 'a'
Kết quả: 3


## 9. Code nộp LeetCode

In [17]:
class Solution:
    def numberOfSubstrings(self, s: str) -> int:
        last = [-1, -1, -1]              # vị trí gần nhất của 'a', 'b', 'c'
        ans = 0
        for j, ch in enumerate(s):
            last[ord(ch) - 97] = j
            ans += min(last) + 1         # số đầu trái i hợp lệ cho đầu phải j
        return ans

sol = Solution()
for s, want in examples:
    assert sol.numberOfSubstrings(s) == want
    print(f"numberOfSubstrings({s!r}) = {sol.numberOfSubstrings(s)}")

numberOfSubstrings('abcabc') = 10
numberOfSubstrings('aaacb') = 3
numberOfSubstrings('abc') = 1


## 10. Vì sao thuật toán đúng

**Cách 3:** mỗi chuỗi con được xác định duy nhất bởi đầu phải `j` và đầu trái `i ≤ j`. Cố định `j`, gọi `p_x = last[x]` là vị trí lớn nhất `≤ j` có ký tự `x` (`−1` nếu không có). `s[i..j]` chứa `x` ⇔ có vị trí `k ∈ [i, j]` với `s[k] = x` ⇔ `p_x ≥ i`. Vậy `s[i..j]` chứa đủ ⇔ `i ≤ min(p_a, p_b, p_c) = m`. Số `i ∈ [0, m]` là `m + 1` (bằng 0 khi `m = −1`). Cộng qua mọi `j` ⇒ đếm mỗi chuỗi con hợp lệ đúng một lần.

**Cách 2:** bất biến — khi vòng `while` kiểm tra, `[left, right]` là cửa sổ có `left` nhỏ nhất chưa được chốt. Do đơn điệu, `right` hiện tại là đầu phải nhỏ nhất làm `left` đủ (nếu một `right` nhỏ hơn đã đủ, `left` đã được chốt từ trước). Vậy mỗi `left` được cộng đúng `n − j(left)` chuỗi con. Các `left` không bao giờ đủ thì không được cộng ✔.

**Cách 5:** công thức bù trừ chuẩn cho "chứa đủ mọi phần tử của `{a,b,c}`" = phần bù của hợp ba biến cố "thiếu `x`"; phần giao "thiếu cả `S`" chính là `N(S)`.

## 11. Phân tích Time / Space

Gọi `n = len(s)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 0. Thô (set mọi chuỗi con) | `O(n³)` | `O(n)` | trọng tài |
| 1. Brute + break | `O(n²)` worst, nhanh khi a/b/c dày | `O(1)` | |
| 2. Cửa sổ trượt | `O(n)` | `O(1)` | mỗi con trỏ ≤ n bước |
| 3. Vị trí cuối | `O(n)` | `O(1)` | 1 vòng, không while |
| 4. Vị trí kế tiếp | `O(n)` | `O(1)` | duyệt ngược |
| 5. Bù trừ | `O(8n)` | `O(1)` | tổng quát cho tập ký tự |

Đáp án tối đa `n(n+1)/2 ≈ 1,25·10^9` — Python không lo tràn số.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
METHODS = {
    "naive": count_naive,
    "brute": count_brute,
    "window": count_window,
    "window_left": count_window_left,
    "last": count_last,
    "next": count_next,
    "incl_excl": count_incl_excl,
    "submit": Solution().numberOfSubstrings,
}
cases = examples + [
    ("aaa", 0),            # thiếu b, c
    ("abab", 0),           # thiếu c
    ("cba", 1),
    ("aabbcc", 4),         # i=0..1 với j=4..5 -> 2*2
    ("abcc", 2),
    ("ccccab", 4),         # chỉ j=5 đóng góp: i = 0..3
    ("acbbcaa", 11),
]
for s, want in cases:
    for name, f in METHODS.items():
        got = f(s)
        assert got == want, (name, s, got, want)
print(f"{len(cases)} ca × {len(METHODS)} cách: đúng hết")

10 ca × 8 cách: đúng hết


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

In [19]:
rng = random.Random(1358)
for _ in range(600):
    n = rng.randint(3, 25)
    alpha = rng.choice(["abc", "aab", "abcc", "aaaabc"])   # thay đổi phân phối ký tự
    s = "".join(rng.choice(alpha) for _ in range(n))
    ref = count_naive(s)
    for name, f in METHODS.items():
        assert f(s) == ref, (name, s)
    # tính chất: đảo ngược chuỗi không đổi đáp án
    assert count_last(s[::-1]) == ref
print("600 chuỗi ngẫu nhiên: mọi cách khớp trọng tài; đảo chuỗi giữ nguyên đáp án")

600 chuỗi ngẫu nhiên: mọi cách khớp trọng tài; đảo chuỗi giữ nguyên đáp án


### 12.3 Kích thước lớn nhất `n = 5·10^4`

In [20]:
rng = random.Random(9)
big = "".join(rng.choice("abc") for _ in range(5 * 10**4))
worst = "a" * (5 * 10**4 - 2) + "bc"
for label, s in [("ngẫu nhiên", big), ("a...abc", worst)]:
    res = []
    for name in ["window", "last", "next", "incl_excl"]:
        st = time.perf_counter()
        res.append(METHODS[name](s))
        print(f"{label:10} {name:10} -> {res[-1]}  ({time.perf_counter() - st:.3f}s)")
    assert len(set(res)) == 1
assert count_last(worst) == 5 * 10**4 - 2

ngẫu nhiên window     -> 1249800505  (0.009s)
ngẫu nhiên last       -> 1249800505  (0.006s)
ngẫu nhiên next       -> 1249800505  (0.006s)
ngẫu nhiên incl_excl  -> 1249800505  (0.017s)
a...abc    window     -> 49998  (0.009s)
a...abc    last       -> 49998  (0.005s)
a...abc    next       -> 49998  (0.005s)


a...abc    incl_excl  -> 49998  (0.014s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khởi tạo `last = [0, 0, 0]` thay vì `-1`

In [21]:
def wrong_last_zero(s):
    last = [0, 0, 0]
    ans = 0
    for j, ch in enumerate(s):
        last[ord(ch) - 97] = j
        ans += min(last) + 1
    return ans

for s, want in examples + [("aaa", 0)]:
    print(f"{s!r:10} -> sai cho {wrong_last_zero(s)}, đúng là {want}")

'abcabc'   -> sai cho 12, đúng là 10
'aaacb'    -> sai cho 7, đúng là 3
'abc'      -> sai cho 3, đúng là 1
'aaa'      -> sai cho 3, đúng là 0


**Vì sao sai:** `0` là một vị trí **hợp lệ** — ký tự chưa xuất hiện bị coi như đã có ở vị trí 0 ⇒ các bước chưa đủ ba ký tự vẫn được cộng (ít nhất 1 mỗi bước), kết quả bị thừa. **Sửa:** khởi tạo `-1` (nghĩa là "chưa thấy").

### Lỗi 2 — Quên `+ 1`

In [22]:
def wrong_no_plus_one(s):
    last = [-1, -1, -1]
    ans = 0
    for j, ch in enumerate(s):
        last[ord(ch) - 97] = j
        ans += max(0, min(last))
    return ans

for s, want in examples:
    print(f"{s!r:10} -> sai cho {wrong_no_plus_one(s)}, đúng là {want}")

'abcabc'   -> sai cho 6, đúng là 10
'aaacb'    -> sai cho 2, đúng là 3
'abc'      -> sai cho 0, đúng là 1


**Vì sao sai:** các đầu trái hợp lệ là `0, 1, …, m` — có `m + 1` giá trị, không phải `m`. **Sửa:** `min(last) + 1`.

### Lỗi 3 — Cửa sổ trượt cộng `right − left + 1` (đếm mọi cửa sổ)

In [23]:
def wrong_window_size(s):
    cnt = [0, 0, 0]
    ans = left = 0
    for right, ch in enumerate(s):
        cnt[ord(ch) - 97] += 1
        while all(cnt):
            cnt[ord(s[left]) - 97] -= 1
            left += 1
        ans += right - left + 1           # đây là số chuỗi con KHÔNG đủ kết thúc tại right
    return ans

for s, want in examples:
    print(f"{s!r:10} -> sai cho {wrong_window_size(s)}, đúng là {want}")

'abcabc'   -> sai cho 11, đúng là 10
'aaacb'    -> sai cho 12, đúng là 3
'abc'      -> sai cho 5, đúng là 1


**Vì sao sai:** sau khi thu hẹp, `[left, right]` là cửa sổ **không đủ** dài nhất; `right − left + 1` là số chuỗi con **thiếu** kết thúc tại `right` (pattern của bài "tối đa k ký tự khác nhau"). **Sửa:** cộng `left` (số đầu trái đủ), hoặc dùng phần bù `tổng − Σ(right − left + 1)`.

### Lỗi 4 — Đếm chuỗi con **phân biệt** bằng `set`

In [24]:
def wrong_distinct(s):
    n = len(s)
    return len({s[i:j + 1] for i in range(n) for j in range(i, n) if len(set(s[i:j + 1])) == 3})

for s, want in examples:
    print(f"{s!r:10} -> sai cho {wrong_distinct(s)}, đúng là {want}")

'abcabc'   -> sai cho 9, đúng là 10
'aaacb'    -> sai cho 3, đúng là 3
'abc'      -> sai cho 1, đúng là 1


**Vì sao sai:** đề đếm theo **vị trí** — ví dụ 1 tính `"abc"` hai lần. **Sửa:** đếm cặp `(i, j)`, không đưa nội dung vào `set`.

### Lỗi 5 — Brute force cộng `1` thay vì `n − j` rồi `break`

In [25]:
def wrong_break(s):
    n = len(s)
    ans = 0
    for i in range(n):
        seen = set()
        for j in range(i, n):
            seen.add(s[j])
            if len(seen) == 3:
                ans += 1            # chỉ đếm chuỗi ngắn nhất
                break
    return ans

for s, want in examples:
    print(f"{s!r:10} -> sai cho {wrong_break(s)}, đúng là {want}")

'abcabc'   -> sai cho 4, đúng là 10
'aaacb'    -> sai cho 3, đúng là 3
'abc'      -> sai cho 1, đúng là 1


**Vì sao sai:** `break` bỏ qua các chuỗi con dài hơn (cũng hợp lệ). **Sửa:** cộng `n − j` trước khi `break` (hoặc không `break` và cộng 1 cho mỗi `j`, nhưng chậm hơn).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm chuỗi con/mảng con thỏa điều kiện đơn điệu"** (đủ thì kéo dài vẫn đủ) ⇒ cửa sổ trượt + cộng theo khối (`n − right` hoặc `left`).
- **"Chứa đủ một tập ký tự nhỏ"** ⇒ mẹo `last[]`: cộng `min(last) + 1` cho mỗi đầu phải.
- **"Tối đa k / ít nhất k"** chuyển đổi được: `#(ít nhất) = tổng − #(không đủ)`.
- Khởi tạo vị trí "chưa thấy" là `-1` (khi duyệt xuôi) hoặc `n` (khi duyệt ngược).
- Viết trọng tài `O(n³)` 1 dòng để test — rất rẻ mà bắt được lỗi lệch một.

## 15. Biến thể của bài

1. **Bảng chữ cái lớn hơn, phải chứa đủ `k` ký tự cho trước** — `last` có `k` phần tử (Bài 3).
2. **Mỗi ký tự xuất hiện ít nhất `t` lần** — lưu `t` vị trí gần nhất của mỗi ký tự (Bài 5).
3. **LeetCode 76 — Minimum Window Substring:** tìm cửa sổ ngắn nhất chứa đủ ký tự của `t`.
4. **LeetCode 992 — Subarrays with K Different Integers:** `#(đúng k) = #(≤ k) − #(≤ k−1)`.
5. **LeetCode 2062 / 2063 — đếm chuỗi con nguyên âm:** cùng kỹ thuật đếm theo vị trí.

In [26]:
def count_contains_all(s, required):
    """Biến thể 1: đếm chuỗi con chứa đủ mọi ký tự trong `required` (chuỗi bất kỳ)."""
    last = {ch: -1 for ch in required}
    ans = 0
    for j, ch in enumerate(s):
        if ch in last:
            last[ch] = j
        ans += min(last.values()) + 1
    return ans

print(count_contains_all("abcabc", "abc"), count_contains_all("xaybz", "ab"), count_contains_all("hello", "lo"))

10 4 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute + break | cố định trái, quét phải | `O(n²)` | `O(1)` | baseline, trọng tài |
| 2. Cửa sổ trượt | hai con trỏ, cộng `n − right` | `O(n)` | `O(1)` | pattern tổng quát, phỏng vấn |
| 3. Vị trí cuối | cộng `min(last) + 1` | `O(n)` | `O(1)` | **ngắn nhất**, nộp bài |
| 4. Vị trí kế tiếp | cộng `n − max(nxt)` | `O(n)` | `O(1)` | theo gợi ý đề |
| 5. Bù trừ | `Σ (−1)^|S| N(S)` | `O(2^k n)` | `O(1)` | khi cần đếm phần bù / tập nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `numberOfSubstrings("aabbcc")`, `numberOfSubstrings("cabca")`.

**Bài 2 (tính tay):** Lập bảng `last` và `ans` cho `s = "bbacab"`.

**Bài 3 (điền TODO):** Hoàn thiện `count_k_letters(s, k)`: `s` gồm các chữ thường, đếm chuỗi con chứa đủ `k` chữ cái đầu tiên của bảng chữ cái (`'a'` … `chr(96+k)`).

**Bài 4 (tự viết lại):** Viết lại Cách 5 (bù trừ) **không** dùng `itertools`, liệt kê tay 8 tập con bằng bitmask.

**Bài 5 (mini challenge):** `count_at_least_t(s, t)`: đếm chuỗi con mà **mỗi** ký tự `a, b, c` xuất hiện **ít nhất `t` lần**, `O(n)`.

In [27]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def count_k_letters(s, k):
    last = [-1] * k
    ans = 0
    for j, ch in enumerate(s):
        # TODO: nếu ch thuộc k chữ đầu thì cập nhật last
        # TODO: cộng min(last) + 1
        pass
    return ans

In [28]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def count_incl_excl_mask(s):
    # TODO: for mask in range(8): banned = {...}; dấu = (-1)^popcount(mask)
    pass

def count_at_least_t(s, t):
    # TODO: lưu danh sách vị trí của mỗi ký tự; đầu trái hợp lệ <= vị trí xuất hiện thứ t tính từ cuối
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Áp dụng `min(last) + 1`. (2) `"aabbcc"`: chỉ các `j = 4, 5` đóng góp. (3) `"cabca"`: đóng góp lần lượt 0, 0, 1, 2, 3.

**Bài 2:** (1) `last` bắt đầu `(-1,-1,-1)`. (2) `j = 3` mới đủ ba ký tự. (3) Cộng các giá trị `min + 1`.

**Bài 3:** (1) `idx = ord(ch) - 97`. (2) `if idx < k: last[idx] = j`. (3) Ký tự ngoài tập không ảnh hưởng `last` nhưng vẫn là một đầu phải hợp lệ.

**Bài 4:** (1) `bin(mask).count("1")` cho số phần tử. (2) `banned = {"abc"[b] for b in range(3) if mask >> b & 1}`. (3) Dùng lại `avoid_count`.

**Bài 5:** (1) `pos[x]` là list các vị trí của `x` đã gặp. (2) Nếu `len(pos[x]) < t` thì chưa có đầu trái hợp lệ. (3) Ngược lại đầu trái phải `≤ pos[x][-t]`; cộng `min(pos[x][-t] for x) + 1`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
print("Bài 1:", count_last("aabbcc"), count_last("cabca"))
assert count_last("aabbcc") == 4 and count_last("cabca") == 6

# Bài 2
print("Bài 2:")
assert trace_last("bbacab") == count_naive("bbacab")

Bài 1: 4 6
Bài 2:
 j | ch | last(a,b,c)  | +min+1 | ans
 0 |  b | (-1, 0, -1)  |      0 | 0
 1 |  b | (-1, 1, -1)  |      0 | 0
 2 |  a | (2, 1, -1)   |      0 | 0
 3 |  c | (2, 1, 3)    |      2 | 2
 4 |  a | (4, 1, 3)    |      2 | 4
 5 |  b | (4, 5, 3)    |      4 | 8


In [30]:
# Bài 3
def count_k_letters_sol(s, k):
    last = [-1] * k
    ans = 0
    for j, ch in enumerate(s):
        idx = ord(ch) - 97
        if idx < k:
            last[idx] = j
        ans += min(last) + 1
    return ans

def count_k_naive(s, k):
    need = set(chr(97 + i) for i in range(k))
    n = len(s)
    return sum(need <= set(s[i:j + 1]) for i in range(n) for j in range(i, n))

rng = random.Random(3)
for _ in range(300):
    k = rng.randint(1, 4)
    s = "".join(rng.choice("abcdef") for _ in range(rng.randint(1, 18)))
    assert count_k_letters_sol(s, k) == count_k_naive(s, k)
assert count_k_letters_sol("abcabc", 3) == 10
print("Bài 3: khớp trọng tài trên 300 chuỗi")

# Bài 4
def count_incl_excl_mask_sol(s):
    ans = 0
    for mask in range(8):
        banned = {"abc"[b] for b in range(3) if mask >> b & 1}
        sign = -1 if bin(mask).count("1") % 2 else 1
        ans += sign * avoid_count(s, banned)
    return ans

for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(3, 20)))
    assert count_incl_excl_mask_sol(s) == count_naive(s)
print("Bài 4: khớp trọng tài trên 300 chuỗi")

Bài 3: khớp trọng tài trên 300 chuỗi
Bài 4: khớp trọng tài trên 300 chuỗi


In [31]:
# Bài 5
def count_at_least_t_sol(s, t):
    pos = {"a": [], "b": [], "c": []}
    ans = 0
    for j, ch in enumerate(s):
        pos[ch].append(j)
        if all(len(p) >= t for p in pos.values()):
            ans += min(p[-t] for p in pos.values()) + 1
    return ans

def count_at_least_t_naive(s, t):
    n = len(s)
    return sum(all(s[i:j + 1].count(x) >= t for x in "abc") for i in range(n) for j in range(i, n))

rng = random.Random(5)
for _ in range(300):
    t = rng.randint(1, 3)
    s = "".join(rng.choice("abc") for _ in range(rng.randint(3, 20)))
    assert count_at_least_t_sol(s, t) == count_at_least_t_naive(s, t)
assert count_at_least_t_sol("abcabc", 1) == 10 and count_at_least_t_sol("abcabc", 2) == 1
print("Bài 5: khớp trọng tài trên 300 chuỗi;", count_at_least_t_sol("aabbccabc", 2))

Bài 5: khớp trọng tài trên 300 chuỗi; 7


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template vị trí cuối:**

```python
last = [-1, -1, -1]
ans = 0
for j, ch in enumerate(s):
    last[ord(ch) - 97] = j
    ans += min(last) + 1
return ans
```

**Template cửa sổ trượt (đếm "ít nhất"):**

```python
for right in range(n):
    thêm s[right]
    while cửa sổ đủ:
        ans += n - right
        bỏ s[left]; left += 1
```

**Checklist trước khi submit:**
- [ ] `last` khởi tạo `-1` (không phải 0).
- [ ] Cộng `min(last) + 1`.
- [ ] Đếm theo vị trí, không theo nội dung.
- [ ] Thử `"abcabc"` → 10, `"aaacb"` → 3, `"aaa"` → 0.
- [ ] `O(n)`, không cắt chuỗi trong vòng lặp.
- [ ] Đúng chữ ký: `numberOfSubstrings(self, s: str) -> int`.